In [3]:
-- Preview the first 10 rows of the survey dataset
-- Useful for understanding column structure and sample values

SELECT *
FROM 'Bus_Ticketing_App_Pre_Launch_Survey.csv'

-- Limit output to 10 records only
LIMIT 10;

,Timestamp,bus_booking_method,booking_frustrations,use_likelihood,fee_willingness,feature_preferences,main_route,preferred_mobile_money,booking _confirmations_preferences,whatsapp
0,4/10/2026 11:32:17,Calling a driver or agent directly,"Uncertainty about departure times, Lack of sea...",I would rarely use it (I prefer the station),Yes,Live Tracking: Seeing exactly where the bus is...,Lusaka — Kitwe / Ndola,Zamtel Kwacha,In-app push notification,j
1,4/10/2026 13:09:44,Physical trip to the station/inter-city terminus,Wasting time/money traveling to the station ju...,I would use it every time I travel,Yes,Refreshments: Pre-ordering a drink or snack fo...,Lusaka — Kitwe / Ndola,Airtel Money,"WhatsApp, SMS/Text Message",0977745548
2,4/10/2026 14:11:59,Physical trip to the station/inter-city terminus,Wasting time/money traveling to the station ju...,I would use it every time I travel,Yes,Live Tracking: Seeing exactly where the bus is...,Lusaka — Livingstone,MTN MoMo,SMS/Text Message,None
3,4/10/2026 14:12:23,Physical trip to the station/inter-city terminus,Poor safety/security at the station,I would likely use it for most trips,Yes,Seat Selection: Choosing exactly where I sit (...,Lusaka — Kitwe / Ndola,Airtel Money,SMS/Text Message,None
4,4/10/2026 14:14:31,Physical trip to the station/inter-city terminus,"Poor safety/security at the station, Lack of s...",I would use it every time I travel,Yes,Seat Selection: Choosing exactly where I sit (...,Lusaka — Kitwe / Ndola,Airtel Money,"SMS/Text Message, Email",None
5,4/10/2026 14:20:17,Physical trip to the station/inter-city terminus,"Uncertainty about departure times, Limited pay...",I would likely use it for most trips,Yes,Seat Selection: Choosing exactly where I sit (...,Lusaka — Kitwe / Ndola,I prefer Bank/Card,SMS/Text Message,None
6,4/10/2026 14:22:03,Physical trip to the station/inter-city terminus,"Lack of seat availability when I arrive, Poor ...",I would use it every time I travel,Yes,Seat Selection: Choosing exactly where I sit (...,Lusaka — Kitwe / Ndola,Airtel Money,"Email, SMS/Text Message, WhatsApp",0979938909
7,4/10/2026 14:23:54,Physical trip to the station/inter-city terminus,Uncertainty about departure times,I might use it depending on the bus operator,Maybe,Live Tracking: Seeing exactly where the bus is...,Lusaka — Kitwe / Ndola,Airtel Money,"In-app push notification, WhatsApp",+260975604824
8,4/10/2026 14:24:25,Physical trip to the station/inter-city terminus,Wasting time/money traveling to the station ju...,I would use it every time I travel,Yes,Travel Protection: Paying an extra K5 – K10 fo...,Lusaka — Solwezi,Airtel Money,"WhatsApp, Email",0974713032
9,4/10/2026 14:29:59,Physical trip to the station/inter-city terminus,Poor safety/security at the station,I would likely use it for most trips,Yes,Seat Selection: Choosing exactly where I sit (...,Lusaka — Kitwe / Ndola,Airtel Money,"WhatsApp, SMS/Text Message",None


In [6]:
-- Create a temporary table (CTE) that splits multiple booking methods
-- stored in one column into separate rows

WITH table1 AS (
    SELECT 
        TRIM(
            UNNEST(
                STRING_TO_ARRAY(
                    bus_booking_method, ','
                )
            )
        ) AS booking_method
    FROM Bus_Ticketing_App_Pre_Launch_Survey.csv
),

-- Count how many times each booking method was selected
table2 AS (
    SELECT 
        booking_method,
        COUNT(*) AS total
    FROM table1
    
    -- Group by booking method
    GROUP BY 1
)

-- Display totals and percentage share of each method
SELECT 
    booking_method,
    total,

    -- Calculate percentage of all booking method selections
    ROUND(
        total * 100.0 / SUM(total) OVER (),
        2
    ) AS total_percentage

FROM table2

-- Show most popular methods first
ORDER BY total_percentage DESC;

,booking_method,total,total_percentage
0,Physical trip to the station/inter-city terminus,98,79.03
1,Calling a driver or agent directly,15,12.10
2,Other,9,7.26
3,Third-party booking office,2,1.61


In [3]:
-- Create a temporary table (CTE) to clean and standardize route names
WITH trips_table AS (

    SELECT 
        CASE 

            -- Replace em dash with normal dash
            WHEN main_route LIKE '%—%' 
                THEN REPLACE(main_route, '—', '-')

            -- Standardize Kasama route names
            WHEN main_route LIKE '%Kasama%' 
                THEN 'Lusaka - Kasama'

            -- Standardize Mongu route names
            WHEN main_route LIKE '%Lusaka to Mongu%' 
              OR main_route LIKE '%MONGU LUSAKA%' 
                THEN 'Lusaka - Mongu'

            -- Standardize Nakonde route names
            WHEN main_route LIKE '%Nakonde%' 
                THEN 'Lusaka - Nakonde'

            -- Standardize Mansa route names
            WHEN main_route LIKE '%Lusaka/Mansa%' 
              OR main_route LIKE '%Lusaka Mansa%' 
                THEN 'Lusaka - Mansa'

            -- Standardize Solwezi route names
            WHEN main_route LIKE '%Solwezi - Lusaka%' 
              OR main_route LIKE '%Lusaka - Solwezi%' 
                THEN 'Lusaka - Solwezi'

            -- Standardize Zambezi route names
            WHEN main_route LIKE '%Zambezi%' 
              OR main_route LIKE '%Solwezi -Lusaka%'
                THEN 'Lusaka - Zambezi'

            -- Standardize Senanga route names
            WHEN main_route LIKE '%Lusaka_senanga%' 
              OR main_route LIKE '%Lusaka Senanga%' 
                THEN 'Lusaka - Senanga'

            -- Keep original value if no match found
            ELSE main_route

        END AS main_route

    FROM Bus_Ticketing_App_Pre_Launch_Survey.csv
),

-- Count trips for each cleaned route
trips_table2 AS (
    SELECT 
        TRIM(main_route) AS routes,
        COUNT(*) AS trips
    FROM trips_table

    -- Group by route
    GROUP BY 1

    -- Keep only routes selected more than once
    HAVING COUNT(*) > 1

    -- Sort by trip count
    ORDER BY 2 DESC
)

-- Show totals and percentage share of each route
SELECT 
    routes,
    trips,

    -- Calculate percentage of total counted trips
    ROUND(
        trips * 100.0 / SUM(trips) OVER (),
        2
    ) AS route_percentage

FROM trips_table2

-- Show most popular routes first
ORDER BY trips DESC;

,routes,trips,route_percentage
0,Lusaka - Kitwe / Ndola,64,64.0
1,Lusaka - Solwezi,16,16.0
2,Lusaka - Livingstone,11,11.0
3,Lusaka - Nakonde,3,3.0
4,Lusaka - Zambezi,2,2.0
5,Lusaka - Mongu,2,2.0
6,Lusaka - Mansa,2,2.0


In [5]:
-- Create a temporary table (CTE) that counts how many respondents
-- selected each preferred mobile money payment option

WITH payment_payment AS (
    SELECT 
        preferred_mobile_money,
        COUNT(*) AS total
    FROM Bus_Ticketing_App_Pre_Launch_Survey.csv
    
    -- Group by payment method
    GROUP BY 1
)

-- Display totals and percentage share for each payment option
SELECT 
    preferred_mobile_money,
    total,

    -- Calculate percentage of total responses
    ROUND(
        total * 100.0 / SUM(total) OVER (),
        2
    ) AS total_percentage

FROM payment_payment

-- Grouping included for compatibility
GROUP BY 1, 2

-- Show highest percentage first
ORDER BY 3 DESC;

,preferred_mobile_money,total,total_percentage
0,Airtel Money,81,73.64
1,MTN MoMo,20,18.18
2,I prefer Bank/Card,7,6.36
3,Zamtel Kwacha,2,1.82


In [7]:
-- Create a temporary table (CTE) that splits multiple frustrations
-- selected in one survey field into separate rows

WITH table3 AS (
    SELECT 
        TRIM(
            UNNEST(
                STRING_TO_ARRAY(
                    booking_frustrations, ','
                )
            )
        ) AS booking_frustrations
    FROM 'Bus_Ticketing_App_Pre_Launch_Survey.csv'
),

-- Count how many times each frustration was selected
table4 AS (
    SELECT 
        booking_frustrations,
        COUNT(*) AS total
    FROM table3
    
    -- Group by each frustration category
    GROUP BY 1
    
    -- Sort by highest count
    ORDER BY 2 DESC
)

-- Display totals and percentage share of each frustration
SELECT 
    booking_frustrations,
    total,

    -- Calculate percentage of all frustration selections
    ROUND(
        total * 100.0 / SUM(total) OVER (),
        2
    ) AS total_percentage

FROM table4

-- Grouping included for compatibility
GROUP BY 1, 2

-- Show highest percentage first
ORDER BY 3 DESC;

,booking_frustrations,total,total_percentage
0,Wasting time/money traveling to the station ju...,77,29.96
1,Uncertainty about departure times,54,21.01
2,Poor safety/security at the station,54,21.01
3,Lack of seat availability when I arrive,36,14.01
4,Limited payment options (cash only),36,14.01


In [8]:
-- Create a temporary table (CTE) that splits multiple selected features
-- from one column into separate rows

WITH table6 AS (
    SELECT 
        TRIM(
            UNNEST(
                STRING_TO_ARRAY(
                    feature_preferences, ','
                )
            )
        ) AS feature_preferences
    FROM 'Bus_Ticketing_App_Pre_Launch_Survey.csv'
),

-- Count how many times each feature was selected
table7 AS (
    SELECT 
        feature_preferences,
        COUNT(*) AS total
    FROM table6
    
    -- Group by each feature option
    GROUP BY 1
    
    -- Sort by popularity
    ORDER BY 2 DESC
)

-- Display counts and percentage share of each feature
SELECT 
    feature_preferences,
    total,

    -- Calculate percentage of all feature selections
    ROUND(
        total * 100.0 / SUM(total) OVER (),
        2
    ) AS total_percentage

FROM table7

-- Grouping included for compatibility
GROUP BY 1, 2

-- Show highest percentage first
ORDER BY 3 DESC;

,feature_preferences,total,total_percentage
0,Seat Selection: Choosing exactly where I sit (...,86,31.27
1,Live Tracking: Seeing exactly where the bus is...,71,25.82
2,Travel Protection: Paying an extra K5 – K10 fo...,70,25.45
3,Refreshments: Pre-ordering a drink or snack fo...,48,17.45


In [9]:
-- Create a temporary table (CTE) that counts how many respondents
-- selected each fee willingness option

WITH fee AS (
    SELECT 
        fee_willingness,
        COUNT(*) AS total
    FROM 'Bus_Ticketing_App_Pre_Launch_Survey.csv'
    
    -- Group by each response category
    GROUP BY 1
    
    -- Sort from highest to lowest count
    ORDER BY 2 DESC
)

-- Display counts and percentage share for each category
SELECT 
    fee_willingness,
    total,

    -- Calculate percentage of total survey responses
    ROUND(
        total * 100.0 / SUM(total) OVER (),
        2
    ) AS total_percentage

FROM fee

-- Grouping included for compatibility
GROUP BY 1, 2

-- Show largest groups first
ORDER BY 2 DESC;

,fee_willingness,total,total_percentage
0,Yes,81,73.64
1,Maybe,22,20.00
2,No,7,6.36


In [10]:
-- Create a temporary table (CTE) that counts how many respondents
-- selected each app usage likelihood option

WITH usability AS (
    SELECT 
        use_likelihood,
        COUNT(*) AS total
    FROM 'Bus_Ticketing_App_Pre_Launch_Survey.csv'
    
    -- Group by each response option
    GROUP BY 1
    
    -- Sort from highest to lowest count
    ORDER BY 2 DESC
)

-- Show counts and percentage share of each option
SELECT 
    use_likelihood,
    total,

    -- Calculate percentage of total responses
    ROUND(
        total * 100.0 / SUM(total) OVER (),
        2
    ) AS total_percentage

FROM usability

-- Grouping included for compatibility
GROUP BY 1, 2

-- Show most common responses first
ORDER BY 2 DESC;

,use_likelihood,total,total_percentage
0,I would use it every time I travel,55,50.00
1,I would likely use it for most trips,35,31.82
2,I might use it depending on the bus operator,17,15.45
3,I would rarely use it (I prefer the station),2,1.82
4,I would never use it,1,0.91


In [22]:
-- Calculate the percentage of unique valid WhatsApp numbers
-- compared to the total number of survey respondents

SELECT 
    ROUND(
        COUNT(DISTINCT whatsapp) * 100.0 / 
        (
            SELECT COUNT(*) 
            FROM 'Bus_Ticketing_App_Pre_Launch_Survey.csv'
        ),
        2
    ) AS whatsapp_percentage
FROM 'Bus_Ticketing_App_Pre_Launch_Survey.csv'

-- Keep only rows where WhatsApp value exists
WHERE whatsapp IS NOT NULL

-- Remove invalid or accidental text responses
AND whatsapp NOT IN (
    'j',
    'Yes, I will be',
    'Do not intend to travel soon'
);

,"round(((count(DISTINCT whatsapp) / (SELECT count_star() FROM ""Bus_Ticketing_App_Pre_Launch_Survey.csv"")) * 100), 2)"
0,37.27


In [4]:
-- Create a temporary table (CTE) that splits multiple communication
-- preferences stored in one column into separate rows
WITH com_table AS (
    SELECT 
        TRIM(
            UNNEST(
                STRING_TO_ARRAY(
                    booking_confirmations_preferences, ','
                )
            )
        ) AS communication
    FROM Bus_Ticketing_App_Pre_Launch_Survey.csv
)

-- Count how many times each communication method was selected
SELECT 
    communication,
    COUNT(*) AS total
FROM com_table

-- Group results by communication method
GROUP BY 1

-- Show most popular methods first
ORDER BY 2 DESC;

,communication,count_star()
0,SMS/Text Message,78
1,WhatsApp,65
2,Email,42
3,In-app push notification,23


In [13]:
-- Count the total number of survey respondents
SELECT 'Survey Respondents', COUNT(*) AS total FROM Bus_Ticketing_App_Pre_Launch_Survey.csv

UNION ALL

-- Count respondents likely to use the app frequently
-- Includes people who said "every time" or "most trips"
SELECT 'Interested in App', COUNT(*) AS total
FROM Bus_Ticketing_App_Pre_Launch_Survey.csv
WHERE use_likelihood LIKE '%every time%' OR use_likelihood LIKE '%most trips%'

UNION ALL

-- Count respondents willing to pay a service fee
SELECT 'Willing to Pay Fee', COUNT(*) AS total
FROM Bus_Ticketing_App_Pre_Launch_Survey.csv
WHERE fee_willingness = 'Yes'

UNION ALL

-- Count respondents who provided their WhatsApp contact
SELECT 'Left WhatsApp', COUNT(*) AS total
FROM Bus_Ticketing_App_Pre_Launch_Survey.csv
WHERE whatsapp IS NOT NULL;

,'Survey Respondents',total
0,Survey Respondents,110
1,Interested in App,90
2,Willing to Pay Fee,81
3,Left WhatsApp,45
